# SportSnap Challenge — Teacher Notebook
**Image classification · Transfer learning & fine-tuning**

### Contains the reference solution. Do not share with students.

| | |
|---|---|
| **Classes** | 100 sports · ~14,000 train images · 500 test images |
| **Metric** | Top-1 accuracy |

### The key ideas
1. **A stronger backbone.** EfficientNet-B0 gives a better frozen-feature starting point than MobileNetV2.
2. **Two-phase training.** First train only the head until it converges, then unfreeze the whole network
and fine-tune at a tiny learning rate (`1e-5`). Unfreezing too early, or with a large learning rate,
wrecks the pretrained weights (*catastrophic forgetting*).
3. **Train long enough** and add **test-time augmentation** (average of the image and its mirror).

### Organizer note — label leakage in the test set
Test file names contain the class (`air hockey_1.jpg`, `ampute football_1.jpg`, ...). A student can reach
100% accuracy by parsing file names, without any model; a perfect leaderboard result very likely
means exactly that. Check the code of top submissions, and rename test files to random IDs
(e.g. `img_0001.jpg`) before the competition is reused.

Needs a GPU (Kaggle: *Settings → Accelerator → GPU*). Runtime ≈ 45–60 min on a T4/P100.

### Notebook structure
1. Setup
2. Explore the data
3. Prepare the data
4. Model: EfficientNet-B0
5. Phase 1 — train the head
6. Phase 2 — fine-tune the whole network
7. Training curves
8. Validation: top-1 / top-5 accuracy
9. Per-class accuracy
10. Most confused sports
11. Submission with test-time augmentation
12. Grading guide

## 1. Setup

In [ ]:
import os
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():   # Apple silicon
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# Works on Kaggle and locally (put the competition files in ./data)
COMPETITION = 'sports-master'
CANDIDATES  = [Path(f'/kaggle/input/competitions/{COMPETITION}'), Path(f'/kaggle/input/{COMPETITION}'), Path('data')]
DATA_DIR    = next(p for p in CANDIDATES if p.exists())
print('Data directory:', DATA_DIR, '→', sorted(os.listdir(DATA_DIR)))

## 2. Explore the Data

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

train_csv = pd.read_csv(DATA_DIR / 'train.csv')
train_csv = train_csv[train_csv['image_id'].map(lambda x: Path(x).suffix.lower() in IMAGE_EXTS)].reset_index(drop=True)

print(f'Training images       : {len(train_csv)}')
print(f'Number of classes     : {train_csv["label"].nunique()}')
print(f'Images per class (avg): {len(train_csv) / train_csv["label"].nunique():.0f}')
train_csv.head()

In [ ]:
sample = train_csv.groupby('label').first().sample(12, random_state=SEED).reset_index()
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for ax, (_, row) in zip(axes.flatten(), sample.iterrows()):
    ax.imshow(Image.open(DATA_DIR / 'train' / row['image_id']))
    ax.set_title(row['label'], fontsize=9, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Prepare the Data

In [ ]:
class SportsDataset(Dataset):
    def __init__(self, df, img_dir, label_map, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.label_map = label_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / row['image_id']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.label_map[row['label']]


class SportsTestDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = self.df.iloc[idx]['image_id']
        img      = Image.open(self.img_dir / image_id).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, image_id


IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

classes      = sorted(train_csv['label'].unique())
label_map    = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in label_map.items()}

train_df, val_df = train_test_split(train_csv, test_size=0.15, stratify=train_csv['label'], random_state=SEED)

train_dataset = SportsDataset(train_df, DATA_DIR / 'train', label_map, transform=train_transform)
val_dataset   = SportsDataset(val_df,   DATA_DIR / 'train', label_map, transform=eval_transform)
test_dataset  = SportsTestDataset(DATA_DIR / 'test.csv', DATA_DIR / 'test', transform=eval_transform)

BATCH_SIZE   = 32
NUM_WORKERS  = 2 if device.type == 'cuda' else 0   # worker processes can't import notebook classes on macOS
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

## 4. Model: EfficientNet-B0

In [ ]:
class SportSnapNet(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        for param in self.backbone.parameters():
            param.requires_grad = False
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)


model = SportSnapNet(num_classes=len(classes)).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters (phase 1): {trainable:,} / {total:,}')

In [ ]:
loss_fn = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = loss_fn(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


history = defaultdict(list)

def run_epochs(n_epochs, optimizer, scheduler, start_epoch=1):
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}")
    print('-' * 55)
    for epoch in range(start_epoch, start_epoch + n_epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer)
        v_loss, v_acc = evaluate(model, val_loader)
        scheduler.step(v_loss)
        for k, v in zip(['train_loss', 'train_acc', 'val_loss', 'val_acc'], [t_loss, t_acc, v_loss, v_acc]):
            history[k].append(v)
        print(f'{epoch:>6} | {t_loss:>10.4f} | {t_acc:>8.1%} | {v_loss:>8.4f} | {v_acc:>6.1%}')

## 5. Phase 1 — Train the Head

In [ ]:
HEAD_EPOCHS = 10
optimizer = Adam(model.backbone.classifier.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
run_epochs(HEAD_EPOCHS, optimizer, scheduler)

## 6. Phase 2 — Fine-Tune the Whole Network

The head has converged, so its gradients are now small and informative. We unfreeze the backbone and
train everything with a learning rate 100× smaller, which nudges the pretrained features towards
sports without destroying them.

In [ ]:
FINETUNE_EPOCHS = 9
for param in model.backbone.parameters():
    param.requires_grad = True

optimizer_ft = Adam(model.parameters(), lr=1e-5)
scheduler_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_ft, patience=2, factor=0.5)
run_epochs(FINETUNE_EPOCHS, optimizer_ft, scheduler_ft, start_epoch=HEAD_EPOCHS + 1)

## 7. Training Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history['train_loss'], 'b-o', label='Train')
ax1.plot(epochs, history['val_loss'],   'r-o', label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')

ax2.plot(epochs, [a * 100 for a in history['train_acc']], 'b-o', label='Train')
ax2.plot(epochs, [a * 100 for a in history['val_acc']],   'r-o', label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy')

for ax in (ax1, ax2):
    ax.axvline(x=HEAD_EPOCHS + 0.5, color='gray', linestyle='--', label='Fine-tune start')
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Validation: Top-1 / Top-5 Accuracy

In [ ]:
@torch.no_grad()
def predict_with_top5(model, loader):
    model.eval()
    top1, top5, total = 0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        top1  += (logits.argmax(1) == labels).sum().item()
        top5  += (logits.topk(5, dim=1).indices == labels[:, None]).any(dim=1).sum().item()
        total += labels.size(0)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return top1 / total, top5 / total, np.array(all_preds), np.array(all_labels)


top1, top5, val_preds, val_labels = predict_with_top5(model, val_loader)
print(f'Val top-1 accuracy: {top1:.1%}')
print(f'Val top-5 accuracy: {top5:.1%}')

## 9. Per-Class Accuracy

In [ ]:
per_class = np.array([(val_preds[val_labels == i] == i).mean() for i in range(len(classes))])
order     = np.argsort(per_class)
colors    = ['green' if a >= 0.8 else 'red' if a < 0.5 else 'orange' for a in per_class[order]]

fig, ax = plt.subplots(figsize=(10, 22))
ax.barh([classes[i] for i in order], per_class[order], color=colors)
ax.axvline(x=0.8, color='black', linestyle='--', linewidth=1.5)
ax.set_xlabel('Accuracy')
ax.set_title('Per-class validation accuracy', fontsize=13, fontweight='bold')
ax.legend(handles=[mpatches.Patch(color='green',  label='≥ 80%'),
                   mpatches.Patch(color='orange', label='50–80%'),
                   mpatches.Patch(color='red',    label='< 50%')], loc='lower right')
plt.tight_layout()
plt.show()

## 10. Most Confused Sports

In [ ]:
cm = confusion_matrix(val_labels, val_preds, labels=range(len(classes)))
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
top_pairs = [divmod(i, len(classes)) for i in np.argsort(off_diag.ravel())[::-1][:10]]

print(f"{'True':25s} -> {'Predicted':25s}  Count")
print('-' * 60)
for t, p in top_pairs:
    print(f'{classes[t]:25s} -> {classes[p]:25s}  {cm[t, p]}')

confused = sorted({c for pair in top_pairs for c in pair})
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm[np.ix_(confused, confused)], annot=True, fmt='d', cmap='Blues',
            xticklabels=[classes[i] for i in confused], yticklabels=[classes[i] for i in confused], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix — most confused sports', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 11. Submission with Test-Time Augmentation

In [ ]:
model.eval()
image_ids, predictions = [], []
with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        probs  = model(images).softmax(1) + model(torch.flip(images, dims=[3])).softmax(1)
        image_ids.extend(ids)
        predictions.extend(probs.argmax(1).cpu().tolist())

submission = pd.DataFrame({'id': image_ids, 'label': predictions})
submission.to_csv('submission.csv', index=False)
print(f'Saved {len(submission)} predictions to submission.csv')
submission.head()

## 12. Grading Guide

### Common mistakes
- Label indices that do not follow the **sorted** class list → predictions land on the wrong classes
- Unfreezing the backbone with `lr=1e-3` → the pretrained weights are wrecked
- Forgetting `model.eval()` at inference (dropout / batch-norm still in training mode)

### What separates strong submissions
1. Understanding frozen vs. fine-tuned transfer learning and why the order matters
2. Reading the training curves to decide when to stop or unfreeze
3. Using the confusion matrix to reason about hard classes (figure-skating variants, gymnastics apparatus)